# Set partition with native FlagQuantum synthesis

Keep the original seeded ten-item dataset, three quantum layers, squared
partition-difference objective, CVaR0.7, 80 iterations and 1000 samples per
optimized/uniform distribution. Angles are radians. All assignments are feasible.
Named array item i is bit i (item0 is LSB); no physical wire arithmetic is needed.


In [ ]:
import matplotlib.pyplot as plt

import flagquantum as fq
from examples.synthesis.applications.optimization.set_partition.overview.plots import (
    partition_figure,
)
from examples.synthesis.applications.optimization.set_partition.overview.program import (
    INITIAL_PARAMETERS,
    cost_layer,
    source_weights,
)
from examples.synthesis.applications.optimization.set_partition.overview.workflow import (
    costs,
    partitions,
    run_workflow,
)
from flagquantum import synth
from flagquantum.synth import (
    RX,
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    qfunc,
)

weights = source_weights()
print("Source seeded data", weights, "total", sum(weights))


@qfunc
def partition_program(
    params: list[float], weights: list[int], v: Output[QArray[QBit]]
) -> None:
    allocate(10, v)
    hadamard_transform(v)
    for layer in range(3):
        cost_layer(params[layer], weights, v)
        for index in range(10):
            RX(params[3 + layer], v[index])


## Synthesize and inspect actual resources

The cost helper expresses the negative square of the weighted signed sum.
Weights are classical integer bindings, rather than a hardcoded dataset. Keep
ten decision bits, zero workspace and explicit operations/memory limits.


In [ ]:
initial_program = synth.synthesize(
    partition_program,
    classical_inputs={"params": INITIAL_PARAMETERS, "weights": weights},
    max_qubits=10,
    max_workspace=0,
    max_operations=5000,
    memory_limit_bytes=4 * 1024**2,
)
initial_readout = dict(
    initial_program.probabilities(fq.run(initial_program), registers=("v",))
)
print("Named assignments", len(initial_readout))
print("Resources", initial_program.report["resources"])


## Optimize and sample actual results

Every evaluation synthesizes and executes. Public Nelder-Mead replaces the
source private optimizer; exact local probabilities replace shot estimation
inside lower-tail CVaR0.7. Final observations are actual seeded CPU samples:
1000 optimized (seed2718) and 1000 uniform (seed31415). Report convergence honestly;
iteration exhaustion and a good sampled assignment can occur together.


In [ ]:
workflow = run_workflow(weights)
print("Initial CVaR", workflow.trace[0].cost)
print("Final CVaR", workflow.optimizer.value)
print("Iterations", workflow.optimizer.iterations)
print("Actual evaluations", workflow.optimizer.evaluations)
print("Converged", workflow.optimizer.converged)
for sample in workflow.samples[:5]:
    print(sample)
print("Optimized observations", sum(row.count for row in workflow.samples))
print("Uniform observations", sum(row.count for row in workflow.uniform))


## Read the best observed partition and plot the source figure roles

Selected means bit1 and other means bit0 (source P2 and P1). Duplicate values
remain different indexed items. The exact reference panel is explicitly labeled;
it does not replace the best observation. Histograms share common bins.


In [ ]:
selected, other = partitions(workflow.best_observed.word, weights)
print("Best actually observed", workflow.best_observed)
print("Selected", selected, "sum", sum(selected))
print("Other", other, "sum", sum(other))
print("Absolute difference", abs(sum(selected) - sum(other)))
partition_plot = partition_figure(workflow)
plt.show()
plt.close(partition_plot)


## Independent bounded reference after sampling

Enumerating1024 assignments replaces external Couenne. This labeled classical
reference never supplies search parameters or observations. A finite sample
hitting an exact optimum does not establish optimizer convergence or advantage.
The source Notebook data differs from the separate frozen Qmod counterpart.


In [ ]:
exact_cost = costs(weights)[workflow.exact_reference_word]
print("Exact bounded reference cost", exact_cost)
print("Exact reference groups", partitions(workflow.exact_reference_word, weights))
print(
    "Optimized observations at exact cost",
    sum(row.count for row in workflow.samples if row.cost == exact_cost),
)
print(
    "Uniform observations at exact cost",
    sum(row.count for row in workflow.uniform if row.cost == exact_cost),
)
